# 04 · Router — one engine, graph + vector

A `RouterQueryEngine` sends a question to the right store: scientific → the arXiv
**graph** engine, current-events → the news **vector** engine. Both run on **one**
shared `AgensEngine` pool.

> Run `01_arxiv_pg/prepare.py` and `03_news_vector_rag/ingest.py` first.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

configure_settings()
from llama_index.core import PropertyGraphIndex, VectorStoreIndex
from llama_index.core.indices.property_graph import VectorContextRetriever
from llama_index.core.query_engine import RetrieverQueryEngine, RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool
llm, embed = get_llm(), get_embed_model()
engine = agens.get_engine()  # ONE shared pool for both stores

In [2]:
# arXiv graph engine
arxiv = agens.make_pg_store("arxiv", vector_dimension=EMBED_DIM, create=False)
graph_idx = PropertyGraphIndex.from_existing(arxiv, embed_model=embed, llm=llm,
                                             kg_extractors=[], use_async=False)
graph_qe = RetrieverQueryEngine.from_args(graph_idx.as_retriever(sub_retrievers=[
    VectorContextRetriever(graph_store=arxiv, embed_model=embed,
                           similarity_top_k=5, path_depth=1, include_text=True)]), llm=llm)

# news vector engine
news = agens.make_vector_store(graph_name="news", node_label="Article")
news_qe = VectorStoreIndex.from_vector_store(news, embed_model=embed).as_query_engine(
    similarity_top_k=5, llm=llm)

router = RouterQueryEngine.from_defaults(
    query_engine_tools=[
        QueryEngineTool.from_defaults(graph_qe, name="arxiv_papers",
            description="scientific/academic questions about papers, authors and methods"),
        QueryEngineTool.from_defaults(news_qe, name="news_articles",
            description="current events: business, technology, world news")],
    selector=LLMSingleSelector.from_defaults(llm=llm), llm=llm, verbose=True)

## A scientific question → the arXiv graph engine

In [3]:
print(router.query("What approaches use neural networks for scientific prediction?"))

Selecting query engine 0: The question pertains to scientific/academic inquiries regarding the use of neural networks in scientific prediction, which aligns with the focus on papers, authors, and methods..


Several approaches utilize neural networks for scientific prediction, particularly in the fields of computer science and statistics. These methods can be found in categories such as neural engineering, artificial intelligence, and machine learning. Specific works authored by researchers in these areas may provide insights into the application of neural networks for predictive tasks in various scientific domains.


## A current-events question → the news vector engine

In [4]:
print(router.query("What are companies doing with artificial intelligence?"))

Selecting query engine 1: The question pertains to current events in business and technology, specifically regarding the application of artificial intelligence by companies..


Companies are utilizing artificial intelligence in various ways, including automating the scanning of websites for suspicious content to help stop predators, enhancing recruitment processes by sorting through resumes and analyzing candidates' responses, and improving customer service by assisting human agents with natural language processing and query prompts. Additionally, AI is being applied in farming to analyze data for maximizing crop yields and in medical diagnosis to customize treatments and interpret medical images. Overall, businesses are optimistic about AI's potential to create job gains and improve operational efficiency, although there is a noted need for investment in reskilling workers to adapt to these changes.


## One shared pool

Both engines draw from the single `AgensEngine` — different graphs, one pool.

In [5]:
rows = news.database_query("SELECT count(*) AS c, state FROM pg_stat_activity "
    "WHERE application_name = 'llama-index-agensgraph' GROUP BY state")
for r in rows: print(f"  {r['c']} connection(s)  state={r['state']}")

  1 connection(s)  state=idle
  1 connection(s)  state=active


In [6]:
agens.close()